In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
import numpy as np

In [7]:
def validate_dataset(csv_path):
    df = pd.read_csv(csv_path)
    
    print("="*50)
    print("📊 DATASET VALIDATION REPORT")
    print("="*50)
    
    # 1. Basic stats
    print(f"\n✅ Total samples: {len(df)}")
    print(f"\n📈 Label distribution:")
    print(df['stress_label'].value_counts())
    print(f"\n📏 Average text length by label:")
    df['word_count'] = df['full_text'].apply(lambda x: len(str(x).split()))
    print(df.groupby('stress_label')['word_count'].mean())
    
    # 2. Check for duplicates
    dupes = df['full_text'].duplicated().sum()
    print(f"\n🔁 Duplicate posts found: {dupes}")
    
    # 3. Check for empty text
    empty = df['full_text'].isna().sum()
    print(f"⚠️ Empty posts found: {empty}")
    
    # 4. Sample 5 random posts from each label
    print("\n" + "="*50)
    print("🔍 RANDOM SAMPLE CHECK (verify manually!)")
    print("="*50)
    for label in ['high', 'medium', 'low']:
        print(f"\n--- {label.upper()} STRESS SAMPLES ---")
        samples = df[df['stress_label'] == label].sample(5)
        for i, row in samples.iterrows():
            print(f"\n[{row['subreddit']}] {row['full_text'][:200]}...")
            print("-"*40)


In [8]:
if __name__ == "__main__":
    csv_path = r"F:\UNIVERSITY\NU 3\semester 2\Machine Intelligence\Project\StressClassifier\data\parsed\parsed_data.csv"
    validate_dataset(csv_path)

📊 DATASET VALIDATION REPORT

✅ Total samples: 140163

📈 Label distribution:
stress_label
high      75000
medium    35163
low       30000
Name: count, dtype: int64

📏 Average text length by label:
stress_label
high      224.928320
low       135.361200
medium    232.367631
Name: word_count, dtype: float64

🔁 Duplicate posts found: 305
⚠️ Empty posts found: 0

🔍 RANDOM SAMPLE CHECK (verify manually!)

--- HIGH STRESS SAMPLES ---

[depression] Depressed anxious bipolar girl needing someone to mic with on skype I'm 16, female, from Canada. I was hoping for someone to talk to me on Skype. I have extreme troubles being alone. My counselor foun...
----------------------------------------

[offmychest] My friends are slowly chipping away at me, and they don't have any clue. Every time they mention their boyfriend or girlfriend, or the new person they're talking to. Every time they tell stories about...
----------------------------------------

[offmychest] Pissed off I was at a goat show today 

In [9]:
def full_verification(csv_path, output_folder):
    df = pd.read_csv(csv_path)
    report = []

    print("="*60)
    print("🔍 FULL DATASET VERIFICATION REPORT")
    print("="*60)

    # ─────────────────────────────────────────
    # 1. BASIC STATS
    # ─────────────────────────────────────────
    print(f"\n📊 Total samples (before cleaning): {len(df)}")
    report.append(f"Total samples before cleaning: {len(df)}")

    # ─────────────────────────────────────────
    # 2. REMOVE DUPLICATES
    # ─────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset='full_text')
    after = len(df)
    dupes_removed = before - after
    print(f"\n🔁 Duplicates removed: {dupes_removed}")
    print(f"✅ Samples after deduplication: {after}")
    report.append(f"Duplicates removed: {dupes_removed}")
    report.append(f"Samples after deduplication: {after}")

    # ─────────────────────────────────────────
    # 3. REMOVE NEAR-DUPLICATES (very similar posts)
    # ─────────────────────────────────────────
    df['text_normalized'] = df['full_text'].str.lower().str.strip()
    df['text_normalized'] = df['text_normalized'].apply(
        lambda x: re.sub(r'\s+', ' ', x)
    )
    before = len(df)
    df = df.drop_duplicates(subset='text_normalized')
    near_dupes = before - len(df)
    df = df.drop(columns=['text_normalized'])
    print(f"🔁 Near-duplicates removed: {near_dupes}")
    report.append(f"Near-duplicates removed: {near_dupes}")

    # ─────────────────────────────────────────
    # 4. LABEL DISTRIBUTION
    # ─────────────────────────────────────────
    print(f"\n📈 Label distribution:")
    dist = df['stress_label'].value_counts()
    print(dist)
    report.append(f"Label distribution: {dist.to_dict()}")

    # Plot distribution
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x='stress_label', 
                  order=['high', 'medium', 'low'],
                  palette=['red', 'orange', 'green'])
    plt.title('Stress Label Distribution')
    plt.savefig(f"{output_folder}/label_distribution.png")
    plt.close()
    print(f"📊 Distribution chart saved!")

    # ─────────────────────────────────────────
    # 5. TEXT LENGTH ANALYSIS
    # ─────────────────────────────────────────
    df['word_count'] = df['full_text'].apply(
        lambda x: len(str(x).split())
    )
    print(f"\n📏 Word count statistics by label:")
    print(df.groupby('stress_label')['word_count'].describe())

    # Plot word count distribution
    plt.figure(figsize=(10, 5))
    for label, color in zip(['high', 'medium', 'low'], 
                             ['red', 'orange', 'green']):
        subset = df[df['stress_label'] == label]['word_count']
        plt.hist(subset, bins=50, alpha=0.5, 
                 label=label, color=color)
    plt.title('Word Count Distribution by Stress Level')
    plt.xlabel('Word Count')
    plt.ylabel('Number of Posts')
    plt.legend()
    plt.savefig(f"{output_folder}/word_count_distribution.png")
    plt.close()
    print(f"📊 Word count chart saved!")

    # ─────────────────────────────────────────
    # 6. TOP KEYWORDS PER CLASS
    # ─────────────────────────────────────────
    print(f"\n🔑 Top 10 keywords per stress level:")
    stopwords = set(['i', 'me', 'my', 'the', 'a', 'an', 'and', 
                     'or', 'but', 'in', 'on', 'at', 'to', 'for',
                     'of', 'with', 'is', 'it', 'this', 'that',
                     'was', 'are', 'be', 'have', 'had', 'has',
                     'do', 'did', 'not', 'he', 'she', 'they',
                     'we', 'you', 'so', 'if', 'just', 'like',
                     'get', 'got', 'can', 'will', 'about'])
    
    for label in ['high', 'medium', 'low']:
        texts = ' '.join(df[df['stress_label'] == label]['full_text'].tolist())
        words = [w.lower() for w in texts.split() 
                 if w.lower() not in stopwords and len(w) > 3]
        top_words = Counter(words).most_common(10)
        print(f"\n  {label.upper()}: {[w for w,c in top_words]}")

    # ─────────────────────────────────────────
    # 7. CROSS-LABEL CONTAMINATION CHECK
    # ─────────────────────────────────────────
    print(f"\n⚠️ Cross-label contamination check:")
    
    high_keywords = ['suicide', 'kill myself', 'end my life', 
                     'want to die', 'hopeless']
    low_keywords  = ['happy', 'great', 'wonderful', 'blessed', 'excited']
    
    # High stress keywords appearing in LOW stress posts
    low_posts = df[df['stress_label'] == 'low']['full_text']
    contaminated_low = low_posts.apply(
        lambda x: any(k in str(x).lower() for k in high_keywords)
    ).sum()
    print(f"  High-stress keywords in LOW posts: {contaminated_low}")

    # Low stress keywords appearing in HIGH stress posts  
    high_posts = df[df['stress_label'] == 'high']['full_text']
    contaminated_high = high_posts.apply(
        lambda x: any(k in str(x).lower() for k in low_keywords)
    ).sum()
    print(f"  Low-stress keywords in HIGH posts: {contaminated_high}")

    # ─────────────────────────────────────────
    # 8. SUBREDDIT COVERAGE
    # ─────────────────────────────────────────
    print(f"\n📂 Posts per subreddit:")
    print(df.groupby(['stress_label', 'subreddit']).size())

    # ─────────────────────────────────────────
    # 9. SAVE CLEAN VERIFIED DATASET
    # ─────────────────────────────────────────
    clean_path = f"{output_folder}/verified_clean_data.csv"
    df.to_csv(clean_path, index=False, encoding='utf-8')
    print(f"\n💾 Clean verified dataset saved to {clean_path}")
    print(f"✅ Final dataset size: {len(df)} posts")

    # Save report
    report_path = f"{output_folder}/verification_report.txt"
    with open(report_path, 'w') as f:
        f.write('\n'.join(report))
    print(f"📄 Report saved to {report_path}")

    return df



In [10]:
if __name__ == "__main__":
    csv_path = r"F:\UNIVERSITY\NU 3\semester 2\Machine Intelligence\Project\StressClassifier\data\parsed\parsed_data.csv"
    output_folder = r"F:\UNIVERSITY\NU 3\semester 2\Machine Intelligence\Project\StressClassifier\data\parsed"
    df = full_verification(csv_path, output_folder)

🔍 FULL DATASET VERIFICATION REPORT

📊 Total samples (before cleaning): 140163

🔁 Duplicates removed: 305
✅ Samples after deduplication: 139858
🔁 Near-duplicates removed: 7

📈 Label distribution:
stress_label
high      74890
medium    35004
low       29957
Name: count, dtype: int64


C:\Users\dell\AppData\Local\Temp\ipykernel_9820\3803376511.py:51: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x='stress_label',


📊 Distribution chart saved!

📏 Word count statistics by label:
                count        mean         std   min    25%    50%    75%  \
stress_label                                                               
high          74890.0  224.959073  121.695484  30.0  124.0  206.0  315.0   
low           29957.0  135.393364   97.859300  30.0   63.0  104.0  177.0   
medium        35004.0  232.546966  118.038067  30.0  137.0  216.0  319.0   

                max  
stress_label         
high          500.0  
low           500.0  
medium        500.0  
📊 Word count chart saved!

🔑 Top 10 keywords per stress level:

  HIGH: ["don't", 'feel', 'know', 'because', 'what', "i've", 'been', 'when', 'want', 'really']

  MEDIUM: ['been', 'work', 'what', 'when', 'really', 'know', 'because', "don't", 'want', 'would']

  LOW: ['been', 'what', 'really', "i've", 'from', "it's", 'some', 'happy', 'time', 'feel']

⚠️ Cross-label contamination check:
  High-stress keywords in LOW posts: 358
  Low-stress keywo